# IntelliFit Nutrition Plan Model — QLoRA Fine-Tuning on Kaggle

**Model**: `Qwen/Qwen2.5-3B-Instruct` + Full fp16 QLoRA (no quant on Kaggle)  
**Task**: Generate personalised 3-day halal nutrition plans in JSON  
**GPU**: T4 ×2 (2 × 16 GB)

## Setup checklist (first run)
1. Upload `nutrition_sft_train.csv` + `nutrition_sft_eval.csv` as a Kaggle Dataset named `intellifit-nutrition-sft`.
2. Attach the dataset to this notebook via **+ Add Data**.
3. Add Kaggle Secret `HF_TOKEN` (HuggingFace write token) — checkpoints auto-push to the Hub.
4. *(Optional)* Add Kaggle Secret `WANDB_API_KEY` for live loss curves.
5. Enable **GPU T4 ×2** and **Internet** in notebook settings.
6. Run all cells — training takes ~2–4 hours on T4 ×2 (full fp16, no quantization).

## How to RESUME after Kaggle session ends
> Kaggle gives ~9 h/session. If training doesn't finish in one go:
>
> **Option A — HuggingFace Hub (recommended, automatic):**  
> Set `HF_TOKEN` secret. Every checkpoint auto-pushes to the Hub.  
> Next session: set `RESUME_CHECKPOINT_SLUG = ''` and keep `HF_RESUME_REPO` pointing to your repo.  
> The notebook will download the last checkpoint and resume automatically.
>
> **Option B — Manual zip download:**  
> Run **Cell 14 (Zip Checkpoint)** before the session ends → download `checkpoint_to_resume.zip` from the Output tab.  
> Next session: upload the zip as a Kaggle Dataset, set `RESUME_CHECKPOINT_SLUG` to that dataset name.  
> The notebook detects and resumes from it automatically.


In [ ]:
# ── Cell 1: Install / upgrade dependencies ──────────────────────────────────
# Kaggle ships older versions; pin exact versions used for local dev.
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

pip('transformers==4.47.0')
pip('trl==0.12.0')
pip('peft==0.13.0')
pip('-U', 'bitsandbytes')
pip('accelerate>=0.30.0')
pip('datasets>=2.20.0')
pip('huggingface_hub>=0.24.0')
pip('wandb')   # optional — for live loss curves

print('✅  Dependencies installed.')

In [ ]:
# ── Cell 2: Core imports ─────────────────────────────────────────────────────
import os, json, glob, zipfile

# Prevent CUDA memory fragmentation (fixes OOM on T4 single-GPU runs)
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, TaskType, PeftModel
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM

print(f'torch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():

    print(f'GPU: {torch.cuda.get_device_name(0)}'
          f'  |  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: Secrets & environment ───────────────────────────────────────────
import os
from kaggle_secrets import UserSecretsClient   # only available on Kaggle

secrets = UserSecretsClient()

# Hugging Face — for pushing the trained adapter + checkpoints to the Hub
try:
    HF_TOKEN = secrets.get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = HF_TOKEN
    PUSH_TO_HUB = True
    print('✅  HF_TOKEN found — checkpoints will be pushed to the Hub automatically.')
    print('    If Kaggle times out, your progress is safe on the Hub!')
except Exception:
    HF_TOKEN = None
    PUSH_TO_HUB = False
    print('⚠️  HF_TOKEN not found — checkpoints saved to /kaggle/working/ ONLY.')
    print('    Run Cell 14 to zip & download before the session ends.')

# Weights & Biases — optional live loss curves
try:
    WANDB_KEY = secrets.get_secret('WANDB_API_KEY')
    os.environ['WANDB_API_KEY'] = WANDB_KEY
    import wandb
    wandb.login(key=WANDB_KEY, relogin=True)
    REPORT_TO = 'wandb'
    print('✅  W&B found — training metrics will stream to wandb.ai')
except Exception:
    REPORT_TO = 'none'
    print('ℹ️   No WANDB_API_KEY — logging to console only.')


In [ ]:
# ── Cell 3: Configuration ────────────────────────────────────────────────────
import os, glob

# ─── Training dataset ────────────────────────────────────────────────────────
print("Looking for dataset files in /kaggle/input...")
train_files = glob.glob('/kaggle/input/**/nutrition_sft_train.csv', recursive=True)
eval_files  = glob.glob('/kaggle/input/**/nutrition_sft_eval.csv', recursive=True)

if not train_files or not eval_files:
    raise FileNotFoundError(
        "Dataset files not found anywhere in /kaggle/input/. "
        "Did you attach 'intellifit-nutrition-sft' to the notebook?"
    )

TRAIN_FILE = train_files[0]
EVAL_FILE  = eval_files[0]

# ─── Model & output ──────────────────────────────────────────────────────────
MODEL_ID       = 'Qwen/Qwen2.5-3B-Instruct'
OUTPUT_DIR     = '/kaggle/working/qwen2.5-3b-nutrition-v1'
HUB_MODEL_ID   = 'youssefeemad/qwen2.5-3b-intellifit-nutrition'  

# ─── Resume config ───────────────────────────────────────────────────────────
HF_RESUME_REPO         = HUB_MODEL_ID   
RESUME_CHECKPOINT_SLUG = 'datasets/youssefeemad/qwen25-3b-nutrition-ckpt-800'

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Train : {TRAIN_FILE}')
print(f'Eval  : {EVAL_FILE}')
print(f'Output: {OUTPUT_DIR}')


In [ ]:

# ── Cell 3b: Detect resume checkpoint ────────────────────────────────────────
import glob, shutil, zipfile, json, os

RESUME_FROM = None

# ── Always print what's mounted in /kaggle/input/ ────────────────────────────
print('=== /kaggle/input/ contents ===')
kaggle_input = '/kaggle/input'
if os.path.exists(kaggle_input):
    for item in sorted(os.listdir(kaggle_input)):
        item_path = os.path.join(kaggle_input, item)
        if os.path.isdir(item_path):
            sub_items = os.listdir(item_path)
            print(f'  📁 {item}/  ({len(sub_items)} items)')
            for s in sorted(sub_items)[:10]:  # show first 10
                s_path = os.path.join(item_path, s)
                size_mb = os.path.getsize(s_path) / 1e6 if os.path.isfile(s_path) else 0
                tag = '📄' if os.path.isfile(s_path) else '📁'
                print(f'       {tag} {s}' + (f'  ({size_mb:.1f} MB)' if os.path.isfile(s_path) else '/'))
        else:
            print(f'  📄 {item}')
else:
    print('  /kaggle/input does not exist')
print('=== End /kaggle/input/ ===\n')

def _find_checkpoint_in(search_dir, is_readonly):
    """Search search_dir for a usable HuggingFace checkpoint.
    Returns (RESUME_FROM path, or None).
    """
    # Strategy 1: checkpoint-N subfolder at root or 1 level deep
    ckpts = sorted(glob.glob(os.path.join(search_dir, 'checkpoint-*')))
    if not ckpts:
        ckpts = sorted(glob.glob(os.path.join(search_dir, '*', 'checkpoint-*')))
    if ckpts:
        src = ckpts[-1]
        print(f'   📁 Found checkpoint subfolder: {src}')
        if is_readonly:
            dst = os.path.join(OUTPUT_DIR, os.path.basename(src))
            print(f'   Copying to writable: {dst} ...')
            shutil.copytree(src, dst, dirs_exist_ok=True)
            return dst
        return src

    # Strategy 2: flat structure — trainer_state.json at root or 1 level deep
    ts_candidates = [os.path.join(search_dir, 'trainer_state.json')]
    ts_candidates += glob.glob(os.path.join(search_dir, '*', 'trainer_state.json'))
    ts_path = next((p for p in ts_candidates if os.path.exists(p)), None)
    if ts_path:
        flat_dir = os.path.dirname(ts_path)
        with open(ts_path) as f:
            step = json.load(f).get('global_step', 0)
        print(f'   📄 Flat checkpoint at {flat_dir} (step={step})')
        ckpt_path = os.path.join(OUTPUT_DIR, f'checkpoint-{step}')
        os.makedirs(ckpt_path, exist_ok=True)
        for fname in os.listdir(flat_dir):
            src_f = os.path.join(flat_dir, fname)
            if os.path.isfile(src_f):
                shutil.copy2(src_f, os.path.join(ckpt_path, fname))
        return ckpt_path

    return None

if RESUME_CHECKPOINT_SLUG:
    resume_input = f'/kaggle/input/{RESUME_CHECKPOINT_SLUG}'

    if not os.path.exists(resume_input):
        print(f'⚠️  Dataset folder not found: {resume_input}')
        print('    → Searching ALL of /kaggle/input/ for any checkpoint ...')
        # Search every mounted dataset for a checkpoint
        for ds in sorted(os.listdir(kaggle_input)):
            ds_path = os.path.join(kaggle_input, ds)
            if not os.path.isdir(ds_path):
                continue
            # skip the training data folder
            if 'intellifit' in ds.lower() or 'sft' in ds.lower():
                continue
            print(f'   Checking {ds_path} ...')
            result = _find_checkpoint_in(ds_path, is_readonly=True)
            if result:
                RESUME_FROM = result
                print(f'✅  Found checkpoint in {ds}: {RESUME_FROM}')
                break
        if RESUME_FROM is None:
            print('   No checkpoint found in any mounted dataset.')
    else:
        # ── Standard path: slug folder exists ────────────────────────────────
        zips = sorted(glob.glob(os.path.join(resume_input, '*.zip')))
        if zips:
            print(f'📦 Extracting {zips[0]} → {OUTPUT_DIR} ...')
            with zipfile.ZipFile(zips[0], 'r') as zf:
                zf.extractall(OUTPUT_DIR)
            search_dir = OUTPUT_DIR
            is_ro = False
        else:
            search_dir = resume_input
            is_ro = True  # /kaggle/input/ is read-only

        result = _find_checkpoint_in(search_dir, is_readonly=is_ro)
        if result:
            RESUME_FROM = result
            print(f'✅  Resume path: {RESUME_FROM}')
        else:
            print(f'⚠️  No checkpoint files found in {search_dir}')

elif PUSH_TO_HUB and HF_RESUME_REPO and 'YOUR_HF_USERNAME' not in HF_RESUME_REPO:
    from huggingface_hub import snapshot_download, HfApi
    try:
        api = HfApi(token=HF_TOKEN)
        ckpt_names = sorted(set(
            f.split('/')[0] for f in api.list_repo_files(HF_RESUME_REPO)
            if f.startswith('checkpoint-')
        ))
        if ckpt_names:
            local_ckpt = os.path.join(OUTPUT_DIR, ckpt_names[-1])
            snapshot_download(repo_id=HF_RESUME_REPO, local_dir=local_ckpt,
                              allow_patterns=[f'{ckpt_names[-1]}/*'], token=HF_TOKEN)
            RESUME_FROM = local_ckpt
    except Exception as e:
        print(f'ℹ️  Hub check skipped ({e})')

print('🆕 Fresh run' if RESUME_FROM is None else f'▶️   Resume path: {RESUME_FROM}')


In [ ]:
# ── Cell 5: Training precision — full fp16 (no quantization) ────────────────
# On Kaggle T4 ×2 (32 GB total) we load in full fp16 — no 4-bit quant needed.
#
# Benefits vs 4-bit NF4:
#   • No quantise/dequantise overhead → ~2-3× faster per step on T4 tensor cores
#   • Higher gradient fidelity → better convergence with r=64 LoRA
#   • Qwen2.5-3B fp16 ≈ 6 GB total → ~3 GB per GPU across 2 × T4
#
# NOTE: For LOCAL INFERENCE on RTX 4050 6 GB, load the BASE model in 4-bit
# and apply the LoRA adapter on top (see the final sanity cell for a snippet).
print("Training mode: full fp16 — Kaggle T4 ×2, no quantization.")

In [ ]:
# ── Cell 6: Load tokenizer ───────────────────────────────────────────────────
print(f'Loading tokenizer from {MODEL_ID} ...')
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    padding_side='right',
    token=HF_TOKEN,
)
tokenizer.pad_token = tokenizer.eos_token


# Build collator that computes loss only on assistant tokens.
# Token IDs avoid the tokenisation-mismatch bug present with raw-string templates.
response_template_ids = tokenizer.encode('<|im_start|>assistant\n', add_special_tokens=False)
collator = DataCollatorForCompletionOnlyLM(response_template_ids, tokenizer=tokenizer)
print('Tokenizer loaded.')

In [ ]:
# ── Cell 7: Load model ───────────────────────────────────────────────────────
print(f'Loading model {MODEL_ID} in full fp16 (no quantization) ...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map='auto',          # spreads across both T4 GPUs automatically
    trust_remote_code=True,
    torch_dtype=torch.float16,
    token=HF_TOKEN,
)
model.config.use_cache = False
model.config.pretraining_tp = 1
print('Model loaded.')

In [ ]:
# ── Cell 8: LoRA config ──────────────────────────────────────────────────────
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=64,
    lora_alpha=128,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_dropout=0.05,
    bias='none',
)
print('LoRA config ready.')

In [ ]:
# ── Cell 9: Load & format dataset ────────────────────────────────────────────
SYSTEM_PROMPT = (
    "You are IntelliFit Nutrition Coach — an expert Egyptian sports nutritionist. "
    "You create personalised 3-day halal nutrition plans (valid JSON, no markdown). "
    "Each plan has 3 unique days, each with breakfast, lunch, dinner, and one snack. "
    "Plans respect the user's health conditions, allergies, fitness goal, and InBody data. "
    "Output ONLY valid JSON in the exact schema requested. "
    "Use Egyptian food names whenever possible. All food is halal. "
    "When the user mentions a previous or recent plan, generate a completely new plan "
    "with different meals, different food items, and different meal compositions — "
    "never repeat the same foods from the previous plan."
)

print('Loading CSV datasets...')
raw = load_dataset('csv', data_files={'train': TRAIN_FILE, 'eval': EVAL_FILE})
print(f'  Train: {len(raw["train"]):,}  |  Eval: {len(raw["eval"]):,}')

def format_messages(example):
    """Parse full messages from JSON-encoded column and apply chat template."""
    messages = json.loads(example['messages'])
    return {'text': tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )}

print('Applying chat template...')
train_ds = raw['train'].map(format_messages, num_proc=4)
eval_ds  = raw['eval'].map(format_messages,  num_proc=4)
# Drop raw 'messages' column from both splits — SFTTrainer only needs 'text'
train_ds = train_ds.remove_columns(['messages'])
eval_ds  = eval_ds.remove_columns(['messages'])   # BUG FIX: must also remove from eval
print('Dataset ready.')


In [ ]:
# ── Cell 10: SFT Training config ─────────────────────────────────────────────
# Kaggle T4 ×2 = 2×16 GB → bigger batches than local RTX 4050 6 GB
# per_device=4, grad_acc=4 → effective batch = 32  (vs 16 locally)
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,          # halved to avoid OOM on fp32 logits (vocab 152k)
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,          # doubled → effective batch = 16 per device (unchanged)
    eval_strategy='steps',
    eval_steps=25,                          # checkpoint every 25 steps (~22 min) — avoids losing session
    save_strategy='steps',
    save_steps=25,                          # checkpoint every 25 steps → /kaggle/working/
    save_total_limit=4,                     # keep last 4 checkpoints
    load_best_model_at_end=True,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    weight_decay=0.01,
    fp16=True,
    bf16=False,                             # T4 does not have native BF16 hardware
    gradient_checkpointing=True,            # saves ~30% VRAM per GPU
    logging_dir=os.path.join(OUTPUT_DIR, 'logs'),
    logging_steps=25,
    report_to=REPORT_TO,                    # 'wandb' or 'none'
    run_name='intellifit-nutrition-v1',
    optim='paged_adamw_8bit',
    max_grad_norm=0.3,
    dataloader_num_workers=4,               # Linux — use more workers for speed
    group_by_length=True,
    # Hub auto-push every save_steps when token is available
    push_to_hub=PUSH_TO_HUB,
    hub_model_id=HUB_MODEL_ID if PUSH_TO_HUB else None,
    hub_token=HF_TOKEN,
    hub_strategy='checkpoint',              # push every checkpoint, not just the end
    # SFT-specific
    max_seq_length=2048,
    dataset_text_field='text',
    packing=False,
)
print('SFTConfig ready.')


In [ ]:
# ── Cell 11: Train ───────────────────────────────────────────────────────────
# Checkpoints are written to /kaggle/working/ every 100 steps.
# If PUSH_TO_HUB=True, each checkpoint is also pushed to the HF Hub
# so even a Kaggle timeout cannot lose more than ~100 steps of work.
print('Starting SFT training...')
trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    peft_config=lora_config,
    args=sft_config,
    tokenizer=tokenizer,
    data_collator=collator,   # compute loss on assistant tokens only
)

trainer.train(resume_from_checkpoint=RESUME_FROM)
print('\nTraining complete.')


In [ ]:
# ── Cell 12: Save adapter + tokenizer ────────────────────────────────────────
# Files saved to /kaggle/working/ are kept in the "Output" tab of this notebook.
# You can download them directly from there after the run finishes.
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'\n✅  LoRA adapter saved  →  {OUTPUT_DIR}')

# List what was saved
import glob
saved = glob.glob(os.path.join(OUTPUT_DIR, '**'), recursive=True)
for p in sorted(saved)[:30]:
    print(' ', p)

In [ ]:
# ── Cell 14: Zip latest checkpoint for download / next-session resume ─────────
# Run this cell BEFORE your session expires to get a downloadable zip.
# Then: Output tab → Download checkpoint_to_resume.zip
#       Upload it as a new Kaggle Dataset, set RESUME_CHECKPOINT_SLUG in Cell 3.

def zip_latest_checkpoint(output_dir=None):
    if output_dir is None:
        output_dir = '/kaggle/working/qwen2.5-3b-nutrition-v1'
    checkpoints = sorted(glob.glob(os.path.join(output_dir, 'checkpoint-*')))
    if not checkpoints:
        print('No checkpoints found yet.')
        return None
    latest   = checkpoints[-1]
    zip_path = '/kaggle/working/checkpoint_to_resume.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(latest):
            for file in files:
                abs_path = os.path.join(root, file)
                arcname  = os.path.relpath(abs_path, output_dir)
                zf.write(abs_path, arcname)
    size_mb = os.path.getsize(zip_path) / 1e6
    print(f'✅  Saved {zip_path}  ({size_mb:.0f} MB)')
    print('    Download from Output tab → upload as Kaggle Dataset to resume next session.')
    return zip_path

zip_latest_checkpoint()


In [ ]:
# ── Cell 13: Final push to Hugging Face Hub ──────────────────────────────────
# Skipped automatically if HF_TOKEN was not configured.
if PUSH_TO_HUB:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.upload_folder(
        folder_path=OUTPUT_DIR,
        repo_id=HUB_MODEL_ID,
        repo_type='model',
        commit_message='Final adapter — IntelliFit nutrition v1',
    )
    print(f'✅  Final adapter pushed to https://huggingface.co/{HUB_MODEL_ID}')
else:
    print('ℹ️   HF push skipped (no HF_TOKEN).  Download from the Output tab instead.')

In [ ]:
# ── Cell 14: Quick sanity inference ─────────────────────────────────────────
# Loads the saved adapter back and runs one sample generation.
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

SYSTEM_PROMPT = (
    'You are a certified Egyptian nutritionist. '
    'Generate valid JSON halal nutrition plans only. '
    'Never include pork, alcohol, or non-halal ingredients.'
)

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, device_map='auto', torch_dtype=torch.float16, token=HF_TOKEN
)
tok = AutoTokenizer.from_pretrained(OUTPUT_DIR)
peft_model = PeftModel.from_pretrained(base, OUTPUT_DIR)
peft_model.eval()

test_msg = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user',   'content': (
        'Create a 3-day halal nutrition plan for a 25-year-old male, '
        'weight 80 kg, height 175 cm, goal: weight_loss, activity: moderate. '
        'Daily calorie target: 2000 kcal. Return only valid JSON.'
    )},
]

prompt = tok.apply_chat_template(test_msg, tokenize=False, add_generation_prompt=True)
inputs = tok(prompt, return_tensors='pt').to(peft_model.device)

with torch.no_grad():
    out = peft_model.generate(**inputs, max_new_tokens=512, temperature=0.3,
                              top_p=0.9, do_sample=True,
                              eos_token_id=tok.eos_token_id,
                              pad_token_id=tok.eos_token_id)

new_tokens = out[0][inputs['input_ids'].shape[1]:]
print(tok.decode(new_tokens, skip_special_tokens=True))

# ── LOCAL INFERENCE (RTX 4050 6 GB) ─────────────────────────────────────────
# After downloading the LoRA adapter from Kaggle output / HuggingFace Hub,
# load it locally with 4-bit quantization on your RTX 4050 6 GB:
#
# from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# from peft import PeftModel
# import torch
#
# local_bnb = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type='nf4',
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True,
# )
# base_local = AutoModelForCausalLM.from_pretrained(
#     'Qwen/Qwen2.5-3B-Instruct',
#     quantization_config=local_bnb,
#     device_map='auto',
#     torch_dtype=torch.float16,
# )
# tok_local = AutoTokenizer.from_pretrained('/path/to/adapter')
# model_local = PeftModel.from_pretrained(base_local, '/path/to/adapter')
# model_local.eval()
